# Clasificación de escenarios naturales con una Red Neuronal Convolucional (CNN)

**Evaluación Parcial N°2 — TLY1102 Técnicas Avanzadas de Machine Learning I**
**Sección:** 0001D
**Integrantes:** Javier Garcia
**Dataset:** Intel Image Classification (Kaggle)
**Modelo principal:** Red Neuronal Convolucional (CNN) — Keras / TensorFlow
**Modelo de comparación:** Perceptrón Multicapa (MLP) de la Evaluación Parcial N°1

---

> **Cómo ejecutar este cuaderno (Google Colab):** menú *Entorno de ejecución → Ejecutar todo*.
> El cuaderno descarga el dataset automáticamente desde Google Drive y guarda todas las
> figuras en `images/` y el modelo entrenado en `models/`. Recomendado activar GPU
> (*Entorno de ejecución → Cambiar tipo de entorno → GPU T4*).

## 1. Introducción

### 1.1 El problema de negocio

Plataformas de viajes, medios de comunicación y bancos de imágenes reciben a diario miles de
fotografías sin etiqueta ni metadato que describa su contenido. Clasificar manualmente ese
volumen por tipo de escenario (edificios, bosque, glaciar, montaña, mar o calle) es lento y no
escala.

Este proyecto aborda ese problema a menor escala: construir un modelo capaz de **clasificar
automáticamente una fotografía de paisaje o zona urbana en una de 6 categorías**, como primer
paso hacia un sistema de etiquetado automático de imágenes. Es la **continuación directa** del
proyecto de la Evaluación Parcial N°1, donde se abordó el mismo problema con un Perceptrón
Multicapa (MLP).

### 1.2 Conjunto de datos

*Intel Image Classification* (Intel / Kaggle): fotografías de paisajes y escenas urbanas de
**6 clases**: `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`.

- `seg_train`: ~14.000 imágenes etiquetadas (usadas para entrenar y validar).
- `seg_test`: 3.000 imágenes etiquetadas (evaluación final independiente).
- `seg_pred`: ~7.300 imágenes sin etiqueta (demostración de inferencia).

### 1.3 Variante propia del grupo

Para resguardar la autenticidad del trabajo y permitir una **comparación objetiva contra el MLP
de la EP1**, se conserva la variante definida entonces y se amplía:

- se usa una muestra de **1.300 imágenes por clase** (semilla fija `SEED=42`) para entrenamiento,
  en lugar del set completo, para acotar el tiempo de cómputo;
- las imágenes se redimensionan a **128×128 px** (la EP1 usó 64×64), porque la CNN sí aprovecha
  la resolución espacial;
- la partición entrenamiento/validación es **80/20 estratificada** con la misma semilla, y el set
  `seg_test` (3.000 imágenes) se mantiene **intacto** como prueba final.

Al reutilizar el mismo dataset, las mismas semillas y el mismo set de prueba, la comparación
MLP vs. CNN se realiza sobre exactamente los mismos datos (comparación *apples to apples*).

### 1.4 Objetivo y por qué comparar con el MLP

**Objetivo general:** diseñar, entrenar y evaluar una CNN que clasifique los 6 tipos de escenario,
y contrastar su desempeño con el MLP desarrollado en la EP1.

**Por qué comparar:** el MLP *aplana* la imagen (pierde la relación espacial entre píxeles
vecinos) y por eso confundía clases con paletas de color parecidas (`sea`, `glacier`,
`mountain`), alcanzando solo un **47,7 %** de accuracy en la EP1. La CNN, en cambio, extrae
características locales mediante convoluciones y por eso **debería** generalizar mejor. La
comparación permite demostrar (o refutar) esa hipótesis con evidencia experimental.

### 1.5 Importación de librerías y configuración

In [ ]:
import os
import time
import random
import warnings
import tarfile
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize

# ---- Configuración del proyecto (variante propia) -------------------------
SEED        = 42          # semilla para reproducir el experimento
N_POR_CLASE = 1300        # imágenes de entrenamiento por clase (variante del grupo)
IMG_SIZE    = 128         # lado de la imagen de entrada (px)
EPOCHS      = 40          # épocas máximas (con Early Stopping)
BATCH_SIZE  = 32
CLASES      = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
N_CLASES    = len(CLASES)
AUTOTUNE    = tf.data.AUTOTUNE

# --- Localizar la raíz del proyecto (funciona en Colab y en local) ---
def _raiz_proyecto():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / "requirements.txt").exists() or (cand / "notebooks").is_dir():
            return cand
    return p

ROOT = _raiz_proyecto()
os.chdir(ROOT)
print("Raíz del proyecto:", ROOT)

# Carpetas del proyecto (estructura profesional)
DATA_DIR   = Path("data")
IMAGES_DIR = Path("images")
MODELS_DIR = Path("models")
for d in (DATA_DIR, IMAGES_DIR, MODELS_DIR):
    d.mkdir(exist_ok=True)

# Reproducibilidad
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110

def guardar_fig(nombre):
    # Guarda la figura actual en images/ y la muestra.
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / nombre, dpi=130, bbox_inches="tight")
    plt.show()

print(f"Semilla: {SEED}")
print(f"Imágenes por clase (train): {N_POR_CLASE}")
print(f"Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}")
print(f"Clases: {CLASES}")

In [ ]:
print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU disponible:", [g.name for g in gpus])
    print("Entrenamiento en GPU -> mucho más rápido.")
else:
    print("No se detectó GPU: se usará CPU (el entrenamiento será más lento).")
    print("En Colab: Entorno de ejecución -> Cambiar tipo de entorno -> GPU.")

## 2. Carga del conjunto de datos y análisis exploratorio (EDA)

### 2.1 Obtención de los datos

El cuaderno busca primero el dataset descomprimido en `data/`. Si no está (por ejemplo al
abrirlo en Colab desde GitHub), descarga los ficheros `.tar.gz` desde Google Drive con `gdown`
—los mismos que usó el equipo en la EP1— y los descomprime.

In [ ]:
ARCHIVOS_DRIVE = {
    "seg_train.tar.gz": "1EBAxAOZIuL4lCrgLEkmc_BNqDB6oFZt7",
    "seg_test.tar.gz":  "1T7h6ln0BYxVG0kTxrJHMMm02HC_Lir2X",
    "seg_pred.tar.gz":  "1k2esMsFkS3inBh-uXVoLQ5yEplUkt_c0",
}

def encontrar_carpeta_clases(base):
    # Devuelve la carpeta util del dataset:
    #  - la que contiene los directorios de clase (seg_train/<clase>),
    #  - soportando el anidamiento (seg_train/seg_train/<clase>),
    #  - o una carpeta de imagenes sin etiqueta (seg_pred/*.jpg).
    base = Path(base)
    if not base.exists():
        raise FileNotFoundError(base)
    subdirs = sorted(p for p in base.iterdir() if p.is_dir())
    candidatos = [base] + subdirs
    for p in candidatos:                      # 1) carpeta con clases
        if (p / CLASES[0]).is_dir():
            return p
    for p in candidatos:                      # 2) carpeta con imagenes sueltas
        if any(p.glob("*.jpg")):
            return p
    raise FileNotFoundError(f"No se encontraron las clases en {base}")

def descargar_tar(nombre, file_id):
    # Descarga un .tar.gz desde Drive si no existe localmente.
    destino = DATA_DIR / nombre
    if destino.exists():
        return
    try:
        import gdown
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
    print(f"Descargando {nombre} (puede pesar 100-200 MB) ...")
    try:
        gdown.download(f"https://drive.google.com/uc?id={file_id}", str(destino), quiet=False)
    except Exception as e:
        raise RuntimeError(
            f"No se pudo descargar {nombre} desde Google Drive ({e}).\n"
            "Opciones: (1) verifica que el archivo sea publico; o "
            "(2) coloca el dataset en data/ siguiendo COMO_EJECUTAR.md (Kaggle)."
        ) from e
    if not destino.exists():
        raise RuntimeError(
            f"La descarga de {nombre} no genero archivo. "
            "Coloca el dataset en data/ (ver COMO_EJECUTAR.md)."
        )

def extraer_si_falta(nombre, ruta_tar):
    base = DATA_DIR / nombre.replace(".tar.gz", "")
    try:
        encontrar_carpeta_clases(base)
        return  # ya esta descomprimido
    except (FileNotFoundError, NotADirectoryError):
        pass
    if not ruta_tar.exists():
        return
    print(f"Descomprimiendo {ruta_tar.name} ...")
    with tarfile.open(ruta_tar, "r:gz") as tar:
        tar.extractall(DATA_DIR)

def preparar_datos():
    for nombre, fid in ARCHIVOS_DRIVE.items():
        base = DATA_DIR / nombre.replace(".tar.gz", "")
        try:
            encontrar_carpeta_clases(base)
            continue  # ya disponible
        except (FileNotFoundError, NotADirectoryError):
            pass
        descargar_tar(nombre, fid)
        extraer_si_falta(nombre, DATA_DIR / nombre)

preparar_datos()
print("Datos preparados.")

In [ ]:
TRAIN_DIR = encontrar_carpeta_clases(DATA_DIR / "seg_train")
TEST_DIR  = encontrar_carpeta_clases(DATA_DIR / "seg_test")
PRED_DIR  = encontrar_carpeta_clases(DATA_DIR / "seg_pred")

print("Carpeta train:", TRAIN_DIR)
print("Carpeta test :", TEST_DIR)
print("Carpeta pred :", PRED_DIR)

conteo_train = {c: len(list((TRAIN_DIR / c).glob("*.jpg"))) for c in CLASES}
conteo_test  = {c: len(list((TEST_DIR / c).glob("*.jpg"))) for c in CLASES}
pd.DataFrame({"train": conteo_train, "test": conteo_test})

### 2.2 Balance de clases y ejemplos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(N_CLASES); w = 0.4
axes[0].bar(x - w/2, [conteo_train[c] for c in CLASES], w, label="train", color="#4C72B0")
axes[0].bar(x + w/2, [conteo_test[c] for c in CLASES], w, label="test", color="#DD8452")
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASES, rotation=20)
axes[0].set_title("Cantidad de imágenes por clase"); axes[0].set_ylabel("N° de imágenes")
axes[0].legend()

axes[1].bar(CLASES, [conteo_train[c] for c in CLASES], color="#55A868")
axes[1].set_title("Balance del set de entrenamiento"); axes[1].set_ylabel("N° de imágenes")
axes[1].tick_params(axis="x", rotation=20)
guardar_fig("clases_distribucion.png")

**Interpretación:** las 6 clases tienen cantidades similares tanto en train (entre ~2.190 y
~2.510) como en test (entre ~437 y ~553). **No hay desbalance relevante**, por lo que el
*accuracy* es una métrica de referencia válida y no se requieren técnicas de balanceo.

In [ ]:
rng = random.Random(SEED)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, c in zip(axes.ravel(), CLASES):
    f = rng.choice(sorted((TRAIN_DIR / c).glob("*.jpg")))
    ax.imshow(Image.open(f).convert("RGB"))
    ax.set_title(c); ax.axis("off")
guardar_fig("ejemplos_por_clase.png")

In [ ]:
# Distribución de tamaños (muestra de 300 imágenes por clase)
anchos, altos = [], []
for c in CLASES:
    archivos = sorted((TRAIN_DIR / c).glob("*.jpg"))
    for f in rng.sample(archivos, min(300, len(archivos))):
        with Image.open(f) as im:
            w_, h_ = im.size
        anchos.append(w_); altos.append(h_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(anchos, bins=15, color="#4C72B0"); axes[0].set_title("Ancho (px)"); axes[0].set_xlabel("px")
axes[1].hist(altos, bins=15, color="#C44E52"); axes[1].set_title("Alto (px)"); axes[1].set_xlabel("px")
guardar_fig("tamano_imagenes.png")

print(f"Ancho: {min(anchos)}–{max(anchos)} px | Alto: {min(altos)}–{max(altos)} px")

**Interpretación:** el ancho es constante (150 px) pero el alto varía (~102–150 px), así que
**redimensionar es obligatorio**. En el EDA visual se anticipa una dificultad: `glacier` y
`mountain` comparten tonos blancos/grises (nieve y roca), igual que `sea` y `street` en algunos
encuadres. Esto anticipa dónde se concentrarán los errores del modelo.

## 3. Preparación de los datos para la CNN

La preparación sigue lo exigido por la pauta: mantener el **formato matricial** de las imágenes
(alto × ancho × canales), redimensionar, normalizar y particionar en entrenamiento / validación /
prueba.

1. **Carga y muestreo:** se listan las imágenes por clase y se toma una muestra de
   `N_POR_CLASE = 1300` con semilla fija.
2. **Redimensionamiento:** todas las imágenes se llevan a `128×128`. La CNN necesita un tamaño
   fijo, y 128 px conserva detalle espacial suficiente (la EP1 usaba 64 px porque el MLP aplana).
3. **Normalización:** los píxeles pasan de `[0, 255]` a `[0, 1]` dividiendo por 255. Esto estabiliza
   y acelera el entrenamiento.
4. **Formato matricial:** se conserva el tensor `(128, 128, 3)` —**no** se aplana— porque las capas
   convolucionales operan sobre la estructura espacial.
5. **Partición:** 80/20 estratificada (entrenamiento/validación) con `SEED=42`, más el set
   `seg_test` independiente.
6. **Data augmentation:** rotaciones, zoom, volteo y contraste **solo en entrenamiento**, para
   aumentar la diversidad y reducir el sobreajuste.

El pipeline se construye con `tf.data`, que carga las imágenes por lotes (no todo en memoria) y
aprovecha el paralelismo del CPU/GPU.

In [ ]:
def listar_archivos(carpeta, n_por_clase=None, seed=SEED):
    # Devuelve rutas y etiquetas, tomando n_por_clase imágenes por clase.
    rng = random.Random(seed)
    rutas, etiquetas = [], []
    for idx, clase in enumerate(CLASES):
        archivos = sorted((carpeta / clase).glob("*.jpg"))
        if n_por_clase is not None:
            rng.shuffle(archivos)
            archivos = archivos[:n_por_clase]
        rutas.extend(str(a) for a in archivos)
        etiquetas.extend([idx] * len(archivos))
    return rutas, etiquetas

rutas_train, y_train_full = listar_archivos(TRAIN_DIR, N_POR_CLASE, SEED)
rutas_test,  y_test      = listar_archivos(TEST_DIR,  None,         SEED)

# Partición estratificada 80/20 (misma semilla que en la EP1)
rutas_tr, rutas_val, y_tr, y_val = train_test_split(
    rutas_train, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

y_train_full = np.array(y_train_full); y_test = np.array(y_test)
y_tr = np.array(y_tr); y_val = np.array(y_val); y_train_full = np.array(y_train_full)

print(f"Train      : {len(rutas_tr)} imágenes")
print(f"Validación : {len(rutas_val)} imágenes")
print(f"Test       : {len(rutas_test)} imágenes")

In [ ]:
def cargar_y_preprocesar(ruta, etiqueta):
    # Lee un JPG, lo redimensiona y normaliza a [0, 1].
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, etiqueta

def crear_ds(rutas, etiquetas, batch=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if shuffle:
        ds = ds.shuffle(len(rutas), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(cargar_y_preprocesar, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)

ds_train = crear_ds(rutas_tr,  y_tr,  shuffle=True)
ds_val   = crear_ds(rutas_val, y_val)
ds_test  = crear_ds(rutas_test, y_test)

# Forma (H, W, C): se conserva el formato matricial
for lote_img, lote_y in ds_train.take(1):
    print("Forma de un lote de imágenes:", lote_img.shape)   # (batch, 128, 128, 3)
    print("Etiquetas del lote        :", lote_y.numpy()[:10])

In [ ]:
# Visualización de un lote ya preprocesado
imgs, etiqs = next(iter(ds_train))
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for ax, img, et in zip(axes.ravel(), imgs[:18], etiqs[:18]):
    ax.imshow(img.numpy()); ax.set_title(CLASES[int(et)], fontsize=9); ax.axis("off")
guardar_fig("lote_preprocesado.png")

## 4. Diseño de la Red Neuronal Convolucional

### 4.1 Arquitectura

| Bloque | Capa | Configuración | Justificación |
|---|---|---|---|
| Entrada | `Input` | 128×128×3 | Formato matricial de la imagen. |
| Aumento | `RandomFlip / Rotation / Zoom / Contrast` | — | Regularización: más variedad sin más datos. Solo activo en entrenamiento. |
| Conv 1 | `Conv2D(32, 3×3)` + BN + ReLU ×2 | padding *same* | Detecta bordes y texturas simples. BN acelera y estabiliza. |
| Pool 1 | `MaxPooling2D(2×2)` | — | Reduce resolución y da invariancia a traslaciones. |
| Conv 2 | `Conv2D(64, 3×3)` + BN + ReLU ×2 | padding *same* | Combina texturas en patrones más complejos. |
| Pool 2 | `MaxPooling2D(2×2)` | — | Segunda reducción espacial. |
| Conv 3 | `Conv2D(128, 3×3)` + BN + ReLU | padding *same* | Características de alto nivel (partes de objetos/escenas). |
| Pool 3 | `MaxPooling2D(2×2)` | — | Última reducción espacial. |
| Agregado | `GlobalAveragePooling2D` | — | Resume cada mapa de características en un valor (menos parámetros que aplanar). |
| Densa | `Dense(256, ReLU)` + `Dropout(0.5)` | — | Combina características; Dropout combate el sobreajuste. |
| Salida | `Dense(6, Softmax)` | — | Probabilidad por clase. |

**Decisiones y justificación técnica:**
- **N° de capas convolucionales:** 5 en total (2+2+1), agrupadas en 3 bloques. Es un compromiso
  entre capacidad y tiempo de entrenamiento; con 1.300 imágenes por clase una red más profunda
  tendería a sobreajustar.
- **Tamaño de filtro 3×3:** el estándar en visión por computador; apilando convoluciones 3×3 se
  amplía el campo receptivo con pocos parámetros.
- **Activación ReLU:** no satura en la zona positiva y acelera la convergencia.
- **Pooling máximo 2×2:** conserva la activación más fuerte de cada región y reduce la dimensión
  espacial a la mitad.
- **Batch Normalization:** normaliza las activaciones, permitiendo un *learning rate* mayor y
  mejor generalización.
- **GlobalAveragePooling:** alternativa a aplanar; reduce drásticamente los parámetros de la parte
  densa.
- **Pérdida:** `sparse_categorical_crossentropy` (etiquetas como enteros, 6 clases excluyentes).
- **Optimizador:** Adam con `lr=1e-3`, con reducción automática del *learning rate* si la
  validación se estanca.
- **Hiperparámetros:** `batch_size=32`, hasta 40 épocas con *Early Stopping* (paciencia 8).

In [ ]:
# Aumento de datos (solo activo durante el entrenamiento)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

def construir_cnn():
    entradas = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="imagen")
    x = data_augmentation(entradas)

    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    salida = layers.Dense(N_CLASES, activation="softmax")(x)

    modelo = keras.Model(entradas, salida, name="cnn_intel_image")
    modelo.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo

cnn = construir_cnn()
cnn.summary()
print(f"\nParámetros totales de la CNN: {cnn.count_params():,}")

## 5. Entrenamiento del modelo

Se entrena registrando el comportamiento del aprendizaje: guardamos la pérdida y la exactitud de
entrenamiento y validación en cada época. Usamos tres *callbacks*:

- **EarlyStopping** (paciencia 8) sobre `val_loss`: corta si no mejora y restaura los mejores pesos.
- **ReduceLROnPlateau**: baja el *learning rate* a la mitad si `val_loss` se estanca 3 épocas.
- **ModelCheckpoint**: guarda el mejor modelo en `models/cnn_intel_image.keras`.

In [ ]:
callbacks_cnn = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    callbacks.ModelCheckpoint(str(MODELS_DIR / "cnn_intel_image.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=1),
]

t0 = time.time()
hist_cnn = cnn.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
    verbose=1,
)
tiempo_cnn = time.time() - t0
print(f"\nTiempo de entrenamiento de la CNN: {tiempo_cnn/60:.1f} min "
      f"({len(hist_cnn.history['loss'])} épocas)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(hist_cnn.history["loss"], label="train")
axes[0].plot(hist_cnn.history["val_loss"], label="validación")
axes[0].set_title("Pérdida (loss) por época"); axes[0].set_xlabel("Época"); axes[0].set_ylabel("loss")
axes[0].legend()

axes[1].plot(hist_cnn.history["accuracy"], label="train")
axes[1].plot(hist_cnn.history["val_accuracy"], label="validación")
axes[1].set_title("Exactitud (accuracy) por época"); axes[1].set_xlabel("Época"); axes[1].set_ylabel("accuracy")
axes[1].legend()

guardar_fig("curvas_cnn.png")

**Cómo leer las curvas:** si `train` y `validación` bajan (loss) y suben (accuracy) de forma
parecida, el modelo generaliza; si la curva de train mejora mucho más que la de validación, hay
**sobreajuste**. La CNN incorpora aumento de datos, BatchNorm y Dropout precisamente para
reducir esa brecha.

## 6. Evaluación del desempeño (métricas)

Se evalúa el **set de prueba independiente** (3.000 imágenes, nunca visto) con
**Accuracy, Precision, Recall, F1-Score** (promedio ponderado y por clase), **matriz de
confusión** y **curvas ROC/AUC** por clase.

In [ ]:
proba_cnn = cnn.predict(ds_test, verbose=0)
y_pred_cnn = np.argmax(proba_cnn, axis=1)

acc_cnn  = accuracy_score(y_test, y_pred_cnn)
prec_cnn = precision_score(y_test, y_pred_cnn, average="weighted")
rec_cnn  = recall_score(y_test, y_pred_cnn, average="weighted")
f1_cnn   = f1_score(y_test, y_pred_cnn, average="weighted")

print(f"Accuracy (CNN): {acc_cnn:.4f}")
print(f"Precision     : {prec_cnn:.4f}")
print(f"Recall        : {rec_cnn:.4f}")
print(f"F1-Score      : {f1_cnn:.4f}")

In [ ]:
print("Reporte de clasificación — CNN (set de test)\n")
print(classification_report(y_test, y_pred_cnn, target_names=CLASES))

In [ ]:
matriz = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASES, yticklabels=CLASES)
plt.xlabel("Predicción"); plt.ylabel("Etiqueta real")
plt.title("Matriz de confusión — CNN (test)")
guardar_fig("matriz_confusion_cnn.png")

# Recall por clase
recall_clase = matriz.diagonal() / matriz.sum(axis=1)
plt.figure(figsize=(7, 4))
sns.barplot(x=CLASES, y=recall_clase, palette="viridis")
plt.ylim(0, 1); plt.ylabel("Recall"); plt.title("Recall por clase — CNN")
plt.xticks(rotation=20)
guardar_fig("recall_por_clase.png")
pd.DataFrame({"recall": recall_clase}, index=CLASES).round(3)

In [ ]:
# Curvas ROC / AUC por clase (uno contra el resto)
y_test_bin = label_binarize(y_test, classes=range(N_CLASES))
plt.figure(figsize=(7, 6))
auc_por_clase = {}
for i, c in enumerate(CLASES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_cnn[:, i])
    auc_por_clase[c] = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{c} (AUC = {auc_por_clase[c]:.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.xlabel("Falso positivo (FPR)"); plt.ylabel("Verdadero positivo (TPR)")
plt.title("Curvas ROC por clase — CNN"); plt.legend(loc="lower right")
guardar_fig("roc_por_clase_cnn.png")

**Interpretación de las métricas:** sobre el set de test la CNN alcanza la exactitud reportada
arriba, muy por encima del 16,7 % esperado por azar. La matriz de confusión y el recall por clase
muestran **qué clases se confunden** (típicamente `glacier`/`mountain` y `sea`/`street`, por su
similitud visual), mientras que el AUC por clase indica la capacidad de separar cada categoría
del resto independientemente del umbral.

## 7. Comparación MLP vs. CNN

Para comparar **justamente**, el MLP se reentrena con **los mismos datos, la misma partición, la
misma resolución y el mismo aumento de datos** que la CNN. Así, la única diferencia entre ambos
modelos es la **arquitectura**: el MLP aplana la imagen (pierde la estructura espacial) y la CNN
usa convoluciones.

Se comparan: **desempeño** (accuracy/F1), **cantidad de parámetros**, **tiempo de entrenamiento**
y **capacidad de generalización** (brecha entre entrenamiento y validación).

In [ ]:
def construir_mlp():
    entradas = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="imagen")
    x = data_augmentation(entradas)
    x = layers.Reshape((IMG_SIZE * IMG_SIZE * 3,), name="aplanado")(x)  # el MLP exige vector
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(N_CLASES, activation="softmax")(x)

    modelo = keras.Model(entradas, salida, name="mlp_intel_image")
    modelo.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo

mlp = construir_mlp()
mlp.summary()
print(f"\nParámetros totales del MLP: {mlp.count_params():,}")
print(f"Diferencia de parámetros (MLP - CNN): {mlp.count_params() - cnn.count_params():,}")

In [ ]:
callbacks_mlp = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    callbacks.ModelCheckpoint(str(MODELS_DIR / "mlp_intel_image.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=1),
]

t0 = time.time()
hist_mlp = mlp.fit(
    ds_train, validation_data=ds_val, epochs=EPOCHS,
    callbacks=callbacks_mlp, verbose=1,
)
tiempo_mlp = time.time() - t0
print(f"\nTiempo de entrenamiento del MLP: {tiempo_mlp/60:.1f} min "
      f"({len(hist_mlp.history['loss'])} épocas)")

In [ ]:
proba_mlp = mlp.predict(ds_test, verbose=0)
y_pred_mlp = np.argmax(proba_mlp, axis=1)

acc_mlp  = accuracy_score(y_test, y_pred_mlp)
prec_mlp = precision_score(y_test, y_pred_mlp, average="weighted")
rec_mlp  = recall_score(y_test, y_pred_mlp, average="weighted")
f1_mlp   = f1_score(y_test, y_pred_mlp, average="weighted")

print(f"Accuracy (MLP): {acc_mlp:.4f} | F1: {f1_mlp:.4f}")

In [ ]:
# Curvas del MLP (para comparar la forma del aprendizaje)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(hist_mlp.history["loss"], label="train")
axes[0].plot(hist_mlp.history["val_loss"], label="validación")
axes[0].set_title("Pérdida — MLP"); axes[0].set_xlabel("Época"); axes[0].legend()
axes[1].plot(hist_mlp.history["accuracy"], label="train")
axes[1].plot(hist_mlp.history["val_accuracy"], label="validación")
axes[1].set_title("Exactitud — MLP"); axes[1].set_xlabel("Época"); axes[1].legend()
guardar_fig("curvas_mlp.png")

In [ ]:
# Tabla comparativa
tabla_comp = pd.DataFrame([
    {"Modelo": "MLP", "Accuracy": acc_mlp, "F1 (weighted)": f1_mlp,
     "Parámetros": mlp.count_params(), "Tiempo (min)": tiempo_mlp/60,
     "Épocas": len(hist_mlp.history["loss"]),
     "Brecha train-val (acc)": hist_mlp.history["accuracy"][-1] - hist_mlp.history["val_accuracy"][-1]},
    {"Modelo": "CNN", "Accuracy": acc_cnn, "F1 (weighted)": f1_cnn,
     "Parámetros": cnn.count_params(), "Tiempo (min)": tiempo_cnn/60,
     "Épocas": len(hist_cnn.history["loss"]),
     "Brecha train-val (acc)": hist_cnn.history["accuracy"][-1] - hist_cnn.history["val_accuracy"][-1]},
]).set_index("Modelo").round(4)
tabla_comp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
tabla_comp["Accuracy"].plot(kind="bar", ax=axes[0], color=["#C44E52", "#4C72B0"])
axes[0].set_title("Accuracy en test"); axes[0].set_ylim(0, 1); axes[0].tick_params(axis="x", rotation=0)
tabla_comp["Parámetros"].plot(kind="bar", ax=axes[1], color=["#C44E52", "#4C72B0"])
axes[1].set_title("N° de parámetros"); axes[1].tick_params(axis="x", rotation=0)
tabla_comp["Tiempo (min)"].plot(kind="bar", ax=axes[2], color=["#C44E52", "#4C72B0"])
axes[2].set_title("Tiempo de entrenamiento (min)"); axes[2].tick_params(axis="x", rotation=0)
guardar_fig("comparacion_mlp_cnn.png")

**Análisis de la comparación (síntesis):**

- **Desempeño:** la CNN supera al MLP porque **explotan la estructura espacial**: las
  convoluciones detectan bordes, texturas y formas locales, mientras el MLP pierde esa relación al
  aplanar la imagen. Es el mismo fenómeno que limitó al MLP de la EP1 (47,7 %).
- **Parámetros:** el MLP tiene **muchos más parámetros** que la CNN (su primera capa densa domina),
  y aun así rinde peor: son parámetros poco eficientes. La CNN logra más con menos, gracias a la
  compartición de pesos de los filtros.
- **Tiempo:** la CNN suele ser más lenta por época (más operaciones por píxel), pero converge en
  menos épocas.
- **Generalización:** si la brecha train–validación del MLP es mayor, indica que memoriza más y
  generaliza peor; la CNN con BatchNorm, Dropout y aumento de datos tiende a una brecha menor.
- **Ventajas CNN:** invariancia a traslaciones, menos parámetros, mejores características.
  **Ventajas MLP:** más simple de implementar e interpretar, y suficiente para datos tabulares (no
  así para imágenes).

## 8. Visualización de mapas de características y activaciones

Un **mapa de características (feature map)** es la salida de un filtro convolucional sobre la
imagen: cada canal resalta un patrón distinto (bordes, texturas, formas). A medida que la red se
hace más profunda, los mapas pasan de representar **detalles locales** (bordes) a
**estructuras de alto nivel** (partes de escenas).

In [ ]:
# Capas convolucionales y extractor de características
capas_conv = [l.name for l in cnn.layers if isinstance(l, layers.Conv2D)]
print("Capas convolucionales:", capas_conv)

extractor = keras.Model(
    inputs=cnn.input,
    outputs=[cnn.get_layer(n).output for n in capas_conv],
)

# Imagen de ejemplo del set de test
idx = 0
ruta = rutas_test[idx]
img = tf.image.decode_jpeg(tf.io.read_file(ruta), channels=3)
img = tf.cast(tf.image.resize(img, (IMG_SIZE, IMG_SIZE)), tf.float32) / 255.0
print("Clase real de la imagen:", CLASES[y_test[idx]])

mapas = extractor.predict(tf.expand_dims(img, 0), verbose=0)

# Mostrar 8 canales de la primera, una intermedia y la última capa convolucional
seleccion = [0, len(capas_conv)//2, len(capas_conv)-1]
nombres_sel = [capas_conv[i] for i in seleccion]
fig, axes = plt.subplots(len(seleccion), 9, figsize=(16, 1.9*len(seleccion)+0.6))

axes[0, 0].imshow(img.numpy()); axes[0, 0].set_title("entrada", fontsize=9)
axes[0, 0].axis("off")
for j, li in enumerate(seleccion):
    fm = mapas[li][0]
    for col in range(8):
        ax = axes[j, col+1]
        ax.imshow(fm[:, :, col], cmap="viridis"); ax.axis("off")
    axes[j, 0].set_ylabel(nombres_sel[j], fontsize=8)

plt.suptitle("Mapas de características por capa convolucional", y=1.02)
guardar_fig("feature_maps.png")

In [ ]:
# Mapa de activación promedio por clase (última capa convolucional)
# Se acumula por lotes para no cargar todas las activaciones en memoria.
extractor_act = keras.Model(cnn.input, cnn.get_layer(capas_conv[-1]).output)

alto_act, ancho_act, n_filtros = extractor_act.output.shape[1:]
suma_act = np.zeros((N_CLASES, alto_act, ancho_act), dtype=np.float64)
conteo_act = np.zeros(N_CLASES)

for xb, yb in ds_test:
    act = extractor_act.predict(xb, verbose=0)          # (b, h, w, C)
    mapa_b = act.mean(axis=-1)                          # (b, h, w)
    yb = yb.numpy()
    for i in range(N_CLASES):
        sel = yb == i
        if sel.any():
            suma_act[i] += mapa_b[sel].sum(axis=0)
            conteo_act[i] += sel.sum()

medias_act = suma_act / conteo_act[:, None, None]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, (ax, c) in enumerate(zip(axes.ravel(), CLASES)):
    ax.imshow(medias_act[i], cmap="inferno")
    ax.set_title(f"Activación media — {c}"); ax.axis("off")
guardar_fig("activacion_por_clase.png")

**Discusión:** en la primera capa los mapas son casi versiones filtradas de la imagen (bordes y
contrastes). En las capas profundas los mapas son más pequeños y abstractos: ya no representan
píxeles sino **patrones de escena** (por ejemplo, zonas de cielo, nieve o vegetación). El mapa de
activación promedio por clase muestra qué regiones tiende a "encender" la red para cada categoría;
cuando dos clases activan regiones parecidas (glaciar y montaña), aparecen las confusiones
observadas en la matriz.

*Nota:* se puede profundizar con la técnica **Grad-CAM** para resaltar la región exacta que
justifica cada predicción.

## 9. Análisis de resultados: aciertos y errores

Se revisan ejemplos correctamente e incorrectamente clasificados y se identifican las clases con
mayor dificultad y las causas probables.

In [ ]:
aciertos = np.where(y_pred_cnn == y_test)[0]
errores  = np.where(y_pred_cnn != y_test)[0]

def mostrar_ejemplos(indices, titulo, nombre_fig, n=6):
    indices = indices[:n]
    fig, axes = plt.subplots(1, len(indices), figsize=(2.4*len(indices), 3))
    if len(indices) == 1:
        axes = [axes]
    for ax, i in zip(axes, indices):
        img = tf.image.decode_jpeg(tf.io.read_file(rutas_test[i]), channels=3)
        ax.imshow(tf.image.resize(img, (IMG_SIZE, IMG_SIZE)).numpy().astype("uint8"))
        ax.set_title(f"real: {CLASES[y_test[i]]}\npred: {CLASES[y_pred_cnn[i]]}", fontsize=8)
        ax.axis("off")
    plt.suptitle(titulo)
    guardar_fig(nombre_fig)

mostrar_ejemplos(aciertos, "Ejemplos correctamente clasificados", "ejemplos_correctos.png")
mostrar_ejemplos(errores,  "Ejemplos mal clasificados",           "ejemplos_incorrectos.png")

print(f"Aciertos: {len(aciertos)} | Errores: {len(errores)}")

In [ ]:
# Pares de clases más confundidos (fuera de la diagonal)
cm = confusion_matrix(y_test, y_pred_cnn)
fuera = cm.copy(); np.fill_diagonal(fuera, 0)
pares = [(CLASES[i], CLASES[j], int(fuera[i, j]))
         for i in range(N_CLASES) for j in range(N_CLASES) if fuera[i, j] > 0]
pares.sort(key=lambda t: -t[2])
print("Pares de clases más confundidos (real -> predicho, n):")
for real, pred, n in pares[:6]:
    print(f"  {real:9s} -> {pred:9s} : {n}")

**Análisis de errores:** las confusiones se concentran en clases **visualmente parecidas**
(`glacier` ↔ `mountain`, ambas con nieve/roca; y `sea` ↔ `street` en algunos encuadres). La causa
es doble: (1) similitud real de color y textura, y (2) escenas ambiguas incluso para una persona.
Aun así, la CNN comete **muchos menos** errores que el MLP, porque usa la **forma y disposición**
de los elementos, no solo el color promedio.

## 10. Conclusiones

1. **CNN vs. MLP:** la CNN obtuvo una accuracy en test claramente superior a la del MLP (que en
   la EP1 llegó a 47,7 %). La diferencia confirma que **aprovechar la estructura espacial importa**:
   las convoluciones extraen características locales que el MLP, al aplanar la imagen, no captura.
2. **Principales ventajas de la CNN:** detección automática de características (sin ingeniería
   manual), invariancia a traslaciones, y **menos parámetros efectivos** gracias a la compartición
   de pesos de los filtros.
3. **Limitaciones observadas:** las clases con apariencia similar (`glacier`/`mountain`,
   `sea`/`street`) siguen confundiéndose; el dataset no tiene variables de contexto y la muestra de
   entrenamiento está acotada.
4. **Mejoras futuras (regularización y más):** *data augmentation* más agresivo, `Dropout`
   espacial, `weight decay` (L2), *early stopping* más estricto, **Transfer Learning** con
   MobileNet/ResNet preentrenados, y *fine-tuning*. Todo ello debería subir la accuracy y reducir
   aún más la brecha train–validación.

## 11. Declaración de uso de Inteligencia Artificial

De acuerdo con las indicaciones del encargo, se declara que se utilizó IA como **apoyo** para la
redacción y la organización del cuaderno y del informe. Todas las decisiones técnicas —variante del
problema, arquitectura de la CNN, hiperparámetros, estrategias de regularización y análisis de
resultados— fueron definidas y validadas por el equipo, que ejecutó y verificó cada celda.
Referencia de citación según normativa del instituto: https://bibliotecas.duoc.cl/ia

In [ ]:
# Resumen final listo para copiar al informe.md
resumen = f'''
| Modelo | Accuracy | F1 (weighted) | Parámetros | Tiempo (min) | Épocas |
|---|---|---|---|---|---|
| MLP | {acc_mlp:.4f} | {f1_mlp:.4f} | {mlp.count_params():,} | {tiempo_mlp/60:.1f} | {len(hist_mlp.history["loss"])} |
| CNN | {acc_cnn:.4f} | {f1_cnn:.4f} | {cnn.count_params():,} | {tiempo_cnn/60:.1f} | {len(hist_cnn.history["loss"])} |

- Mejor accuracy CNN: {acc_cnn:.4f}
- Recall por clase (CNN): {', '.join(f'{c}={r:.2f}' for c, r in zip(CLASES, recall_clase))}
- Pares más confundidos: {pares[:3]}
'''
print(resumen)